# 07 — Time Series Change Detection

Temporal analysis of NISAR GCOV backscatter stacks: build data cubes,
compute Coefficient of Variation (CoV), run CUSUM change-point detection,
apply backscatter thresholding for inundation mapping, and fit harmonic
models for seasonal baselines.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bullocke/nice-sar/blob/main/notebooks/07_timeseries_change.ipynb)

## 1. Install and Import

In [ ]:
%pip install -q "fsspec<=2025.3.0" "s3fs<=2025.3.0"
%pip install -q --force-reinstall --no-deps "git+https://github.com/bullocke/nice-sar.git@main"

In [ ]:
import logging
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

from nice_sar.io.products import read_gcov
from nice_sar.analysis.timeseries import (
    cusum,
    coefficient_of_variation,
    backscatter_threshold,
    harmonic_fit,
)
from nice_sar.preprocess.multilook import multilook
from nice_sar.preprocess.calibration import linear_to_db

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

## 2. Build a Temporal Data Cube

Stack a time series of GCOV products into a 3-D array `(T, H, W)`.
All NISAR GCOV products on the same track/frame share the same grid,
so no resampling is needed.

In [ ]:
# List of GCOV file paths (replace with real paths or S3 URIs)
gcov_files = sorted(Path("data/gcov/").glob("NISAR_L2_PR_GCOV_*.h5"))
print(f"Found {len(gcov_files)} GCOV files")

# Read HH backscatter from each
arrays = []
for f in gcov_files:
    da_xr = read_gcov(f, polarization="HH")
    arrays.append(da_xr.values)

stack_hh = np.stack(arrays, axis=0)  # (T, H, W)
print(f"Stack shape: {stack_hh.shape}")

# Optional: multilook to 240 m for noise reduction
# stack_hh = np.stack([multilook(a, 8, 8) for a in arrays], axis=0)

## 3. Coefficient of Variation (CoV)

The temporal CoV ($\sigma / \mu$) highlights pixels with high variability
over time — useful for identifying agriculture, inundation dynamics, or
active disturbance.

In [ ]:
cov_map = coefficient_of_variation(stack_hh)

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(cov_map, cmap="YlOrRd", vmin=0, vmax=1.5)
ax.set_title("Temporal Coefficient of Variation (HH)")
plt.colorbar(im, ax=ax, label="CoV")
plt.show()

## 4. CUSUM Change-Point Detection

The Cumulative Sum (CUSUM) method detects the timing and magnitude of
abrupt changes by accumulating residuals from a baseline mean.

- Use the first half of the stack as the stable baseline
- The position of maximum |CUSUM| indicates when change occurred
- Threshold to filter small fluctuations

In [ ]:
# Convert to dB for more normal residual distribution
stack_db = linear_to_db(stack_hh)

result = cusum(stack_db, baseline_count=len(gcov_files) // 2, threshold=3.0)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

im0 = axes[0].imshow(result.magnitude, cmap="hot")
axes[0].set_title("CUSUM Magnitude")
plt.colorbar(im0, ax=axes[0])

change_vis = result.change_index.astype(float)
change_vis[result.change_index == -1] = np.nan  # No change → transparent
im1 = axes[1].imshow(change_vis, cmap="viridis")
axes[1].set_title("Change Timing (time step index)")
plt.colorbar(im1, ax=axes[1])

plt.tight_layout()
plt.show()

In [ ]:
# Plot CUSUM trajectory for a single pixel
row, col = 30, 30
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(result.cusum_series[:, row, col], marker="o", markersize=3)
ax.axhline(0, color="gray", linestyle="--")
ax.set_xlabel("Time step")
ax.set_ylabel("Cumulative Sum")
ax.set_title(f"CUSUM series at pixel ({row}, {col})")
plt.show()

## 5. Backscatter Thresholding (Inundation)

Classify inundation using backscatter intensity thresholds.
- Open water: HH × HV < 5×10⁻⁵ (linear)
- Inundated vegetation: mean HH > 0.5 or HH/HV > 8

In [ ]:
# Simple low-backscatter = open water
water_result = backscatter_threshold(
    stack_hh, low=0.0, high=0.001, min_fraction=0.5
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].imshow(water_result.fraction, cmap="Blues", vmin=0, vmax=1)
axes[0].set_title("Temporal fraction below threshold")

axes[1].imshow(water_result.mask, cmap="Blues")
axes[1].set_title("Open water mask (>50% of dates)")

plt.tight_layout()
plt.show()

## 6. Harmonic Fitting

Fit a first-order harmonic model to capture seasonal variability:

$$y(t) = a_0 + a_1 \cos\left(\frac{2\pi t}{P}\right) + a_2 \sin\left(\frac{2\pi t}{P}\right)$$

Residuals from the harmonic fit can be used for anomaly detection
(e.g., z-scores or Bayesian updating).

In [ ]:
# Day-of-year for each acquisition (replace with real dates)
times = np.linspace(0, 365.25 * 2, len(gcov_files))  # ~2 years

harm = harmonic_fit(stack_db, times, period=365.25)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

im0 = axes[0].imshow(harm.intercept, cmap="viridis")
axes[0].set_title("Mean backscatter (dB)")
plt.colorbar(im0, ax=axes[0])

im1 = axes[1].imshow(harm.amplitude, cmap="YlOrRd")
axes[1].set_title("Seasonal amplitude (dB)")
plt.colorbar(im1, ax=axes[1])

im2 = axes[2].imshow(harm.rmse, cmap="hot")
axes[2].set_title("RMSE (dB)")
plt.colorbar(im2, ax=axes[2])

plt.tight_layout()
plt.show()

In [ ]:
# Plot observed vs fitted time series for a single pixel
row, col = 30, 30
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(times, stack_db[:, row, col], "ko", markersize=3, label="Observed")
fitted = (
    harm.intercept[row, col]
    + harm.amplitude[row, col]
    * np.cos(2 * np.pi * times / 365.25 - harm.phase[row, col])
)
ax.plot(times, fitted, "r-", label="Harmonic fit")
ax.set_xlabel("Day of year")
ax.set_ylabel("Backscatter (dB)")
ax.set_title(f"Harmonic fit at pixel ({row}, {col})")
ax.legend()
plt.show()

## 7. Export Results

In [ ]:
from nice_sar.io.geotiff import write_geotiff
from pyproj import CRS
from rasterio.transform import Affine

# Use attributes from the first GCOV
da_ref = read_gcov(gcov_files[0], polarization="HH")
crs = CRS.from_user_input(da_ref.attrs["crs"])
transform = Affine(*da_ref.attrs["transform"])

write_geotiff("cov_map.tif", cov_map, crs=crs, transform=transform)
write_geotiff("cusum_magnitude.tif", result.magnitude, crs=crs, transform=transform)
write_geotiff("harmonic_amplitude.tif", harm.amplitude, crs=crs, transform=transform)
logger.info("Exported time series results")